In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/3rd-wear-dataset-challenge-hasca-2026/sample_submission.csv
/kaggle/input/competitions/3rd-wear-dataset-challenge-hasca-2026/participant_meta_data.txt
/kaggle/input/competitions/3rd-wear-dataset-challenge-hasca-2026/test/test_inertial_data.npy
/kaggle/input/competitions/3rd-wear-dataset-challenge-hasca-2026/test/test_meta_data.csv
/kaggle/input/competitions/3rd-wear-dataset-challenge-hasca-2026/test/test_videomae_data.npy
/kaggle/input/competitions/3rd-wear-dataset-challenge-hasca-2026/train/inertial_feat/sbj_13.csv
/kaggle/input/competitions/3rd-wear-dataset-challenge-hasca-2026/train/inertial_feat/sbj_17.csv
/kaggle/input/competitions/3rd-wear-dataset-challenge-hasca-2026/train/inertial_feat/sbj_15.csv
/kaggle/input/competitions/3rd-wear-dataset-challenge-hasca-2026/train/inertial_feat/sbj_16.csv
/kaggle/input/competitions/3rd-wear-dataset-challenge-hasca-2026/train/inertial_feat/sbj_19.csv
/kaggle/input/competitions/3rd-wear-dataset-challenge-hasca-2026/tr

In [6]:
# ============================================================
# EXPERIMENT 11 — CROSS-MODAL ATTENTION
# CELL 1: IMPORTS + CONFIGURATION
# ============================================================

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report


# ============================================================
# PATHS
# ============================================================

BASE = "/kaggle/input/competitions/3rd-wear-dataset-challenge-hasca-2026"

TRAIN_INERTIAL_DIR = BASE + "/train/inertial_feat"
TRAIN_VIDEO_DIR = BASE + "/train/videomae_feat"

TEST_INERTIAL_PATH = BASE + "/test/test_inertial_data.npy"
TEST_VIDEO_PATH = BASE + "/test/test_videomae_data.npy"
TEST_META_PATH = BASE + "/test/test_meta_data.csv"

SAMPLE_SUB_PATH = BASE + "/sample_submission.csv"

OUT_DIR = "/kaggle/working"


# ============================================================
# CONFIG
# ============================================================

N_FOLDS = 5

BATCH_SIZE = 256
EPOCHS = 30

LR = 1e-3
WEIGHT_DECAY = 1e-4

NUM_CLASSES = 19

WINDOW_SIZE = 50
STRIDE = 25

INERTIAL_HZ = 50
VIDEO_FPS = 30

VIDEO_WINDOW_FRAMES = 30
VIDEO_MODEL_FRAMES = 15

CHECKPOINT_PREFIX = "exp11_crossmodal"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


print("=" * 70)
print("EXPERIMENT 11 — CROSS-MODAL ATTENTION")
print("=" * 70)

print("Device:", DEVICE)
print("Classes:", NUM_CLASSES)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Learning rate:", LR)

EXPERIMENT 11 — CROSS-MODAL ATTENTION
Device: cuda
Classes: 19
Batch size: 256
Epochs: 30
Learning rate: 0.001


In [7]:
# ============================================================
# CELL 2: LABELS + SENSOR LOCATIONS
# ============================================================

LOCATIONS = [
    "right_arm",
    "right_leg",
    "left_leg",
    "left_arm"
]


LABEL_MAP = {
    "null": 0,
    "jogging": 1,
    "jogging (rotating arms)": 2,
    "jogging (skipping)": 3,
    "jogging (sidesteps)": 4,
    "jogging (butt-kicks)": 5,
    "stretching (triceps)": 6,
    "stretching (lunging)": 7,
    "stretching (shoulders)": 8,
    "stretching (hamstrings)": 9,
    "stretching (lumbar rotation)": 10,
    "push-ups": 11,
    "push-ups (complex)": 12,
    "sit-ups": 13,
    "sit-ups (complex)": 14,
    "burpees": 15,
    "lunges": 16,
    "lunges (complex)": 17,
    "bench-dips": 18
}


LOCATION_MAP = {
    "right_arm": 0,
    "right_leg": 1,
    "left_leg": 2,
    "left_arm": 3
}


SENSOR_COLUMNS = {
    loc: [
        f"{loc}_acc_x",
        f"{loc}_acc_y",
        f"{loc}_acc_z"
    ]
    for loc in LOCATIONS
}


print("Number of classes:", len(LABEL_MAP))
print("Number of sensor locations:", len(LOCATION_MAP))

print("\nLabels:")
for name, idx in LABEL_MAP.items():
    print(idx, "->", name)

Number of classes: 19
Number of sensor locations: 4

Labels:
0 -> null
1 -> jogging
2 -> jogging (rotating arms)
3 -> jogging (skipping)
4 -> jogging (sidesteps)
5 -> jogging (butt-kicks)
6 -> stretching (triceps)
7 -> stretching (lunging)
8 -> stretching (shoulders)
9 -> stretching (hamstrings)
10 -> stretching (lumbar rotation)
11 -> push-ups
12 -> push-ups (complex)
13 -> sit-ups
14 -> sit-ups (complex)
15 -> burpees
16 -> lunges
17 -> lunges (complex)
18 -> bench-dips


In [8]:
# ============================================================
# CELL 3: RANDOM SEED
# ============================================================

def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)

print("Random seed set to 42")

Random seed set to 42


In [9]:
# ============================================================
# CELL 3: RANDOM SEED
# ============================================================

def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)

print("Random seed set to 42")

Random seed set to 42


In [10]:
# ============================================================
# CELL 5: CREATE 1-SECOND WINDOWS
# ============================================================

def make_windows(
    n_rows,
    window_size=WINDOW_SIZE,
    stride=STRIDE
):

    starts = np.arange(
        0,
        n_rows - window_size + 1,
        stride
    )

    return [
        (int(start), int(start + window_size))
        for start in starts
    ]


def window_mode_label(label_ids):

    counts = np.bincount(
        label_ids,
        minlength=NUM_CLASSES
    )

    return int(np.argmax(counts))


print("Window size:", WINDOW_SIZE)
print("Stride:", STRIDE)
print("Overlap: 50%")

Window size: 50
Stride: 25
Overlap: 50%


In [11]:
# ============================================================
# CELL 6: BUILD WINDOWED SAMPLES
# ============================================================

def build_windows_for_subject(
    df,
    sbj_file_id
):

    sbj_id = int(
        df["sbj_id"].iloc[0]
    )

    label_ids = df["label_id"].values

    windows = make_windows(
        len(df)
    )


    samples = []


    # --------------------------------------------------------
    # Convert sensor data to numpy
    # --------------------------------------------------------

    loc_arrays = {
        loc: df[cols].values.astype(np.float32)
        for loc, cols in SENSOR_COLUMNS.items()
    }


    # --------------------------------------------------------
    # Create windows
    # --------------------------------------------------------

    for start, end in windows:

        y = window_mode_label(
            label_ids[start:end]
        )


        # One sample per sensor location

        for loc in LOCATIONS:

            x = loc_arrays[loc][start:end]

            samples.append({
                "x": x,
                "y": y,
                "location": loc,
                "sbj_id": sbj_id,
                "sbj_file": sbj_file_id,
                "start_idx": start
            })


    return samples


print("Window builder ready.")

Window builder ready.


In [12]:
# ============================================================
# CELL 7: BUILD COMPLETE TRAINING DATASET
# ============================================================

def build_dataset_from_dir(
    train_dir
):

    all_samples = []

    files = sorted(
        os.listdir(train_dir)
    )


    for fname in files:

        if not fname.endswith(".csv"):
            continue


        path = os.path.join(
            train_dir,
            fname
        )


        df = load_and_encode(
            path
        )


        sbj_file_id = fname.replace(
            ".csv",
            ""
        )


        samples = build_windows_for_subject(
            df,
            sbj_file_id
        )


        all_samples.extend(
            samples
        )


        print(
            f"{fname}: "
            f"{len(df)} rows -> "
            f"{len(samples)//4} windows -> "
            f"{len(samples)} samples"
        )


    return all_samples


print("=" * 70)
print("BUILDING TRAINING SAMPLES")
print("=" * 70)

samples = build_dataset_from_dir(
    TRAIN_INERTIAL_DIR
)


print("\nTotal samples:", len(samples))

print(
    "Unique subjects:",
    len(
        np.unique(
            [s["sbj_id"] for s in samples]
        )
    )
)

BUILDING TRAINING SAMPLES
sbj_0.csv: 139725 rows -> 5588 windows -> 22352 samples
sbj_0_2.csv: 112050 rows -> 4481 windows -> 17924 samples
sbj_1.csv: 138900 rows -> 5555 windows -> 22220 samples
Cleaning 154176 bad sensor values
sbj_10.csv: 199250 rows -> 7969 windows -> 31876 samples
sbj_11.csv: 111100 rows -> 4443 windows -> 17772 samples
sbj_12.csv: 163500 rows -> 6539 windows -> 26156 samples
sbj_13.csv: 198900 rows -> 7955 windows -> 31820 samples
sbj_14.csv: 160050 rows -> 6401 windows -> 25604 samples
sbj_14_2.csv: 167750 rows -> 6709 windows -> 26836 samples
sbj_15.csv: 130150 rows -> 5205 windows -> 20820 samples
sbj_16.csv: 175500 rows -> 7019 windows -> 28076 samples
sbj_17.csv: 154550 rows -> 6181 windows -> 24724 samples
sbj_18.csv: 131600 rows -> 5263 windows -> 21052 samples
sbj_19.csv: 105450 rows -> 4217 windows -> 16868 samples
sbj_2.csv: 178200 rows -> 7127 windows -> 28508 samples
sbj_20.csv: 95400 rows -> 3815 windows -> 15260 samples
sbj_21.csv: 127800 rows -> 51

In [13]:
# ============================================================
# CELL 8: VIDEOMAE CACHE
# ============================================================

video_cache = {}


def get_video_array(sbj_file):

    if sbj_file not in video_cache:

        video_path = os.path.join(
            TRAIN_VIDEO_DIR,
            sbj_file + ".npy"
        )


        video_cache[sbj_file] = np.load(
            video_path,
            mmap_mode="r"
        )


    return video_cache[sbj_file]


print("Video cache ready.")

Video cache ready.


In [14]:
# ============================================================
# CELL 9: MULTIMODAL PYTORCH DATASET
# ============================================================

class MultimodalWEARDataset(Dataset):

    def __init__(
        self,
        samples,
        mean=None,
        std=None
    ):

        self.samples = samples

        self.mean = mean
        self.std = std


    def __len__(self):

        return len(self.samples)


    def __getitem__(self, index):

        sample = self.samples[index]


        # ====================================================
        # INERTIAL
        # ====================================================

        x = sample["x"].copy()


        if self.mean is not None:

            x = (
                x - self.mean
            ) / self.std


        # [50,3] -> [3,50]

        x = torch.tensor(
            x.T,
            dtype=torch.float32
        )


        # ====================================================
        # VIDEO
        # ====================================================

        sbj_file = sample["sbj_file"]

        video_array = get_video_array(
            sbj_file
        )


        # 50 Hz -> 30 FPS

        video_start = int(
            round(
                sample["start_idx"]
                * VIDEO_FPS
                / INERTIAL_HZ
            )
        )


        # 30-frame window

        video_30 = video_array[
            video_start:
            video_start + VIDEO_WINDOW_FRAMES
        ]


        # 30 -> 15 frames

        video = video_30[::2]


        if video.shape[0] != VIDEO_MODEL_FRAMES:

            raise RuntimeError(
                f"Expected "
                f"{VIDEO_MODEL_FRAMES} video frames, "
                f"got {video.shape}"
            )


        video = torch.tensor(
            np.asarray(video),
            dtype=torch.float32
        )


        # ====================================================
        # LABEL
        # ====================================================

        y = torch.tensor(
            sample["y"],
            dtype=torch.long
        )


        # ====================================================
        # SENSOR LOCATION
        # ====================================================

        location = torch.tensor(
            LOCATION_MAP[
                sample["location"]
            ],
            dtype=torch.long
        )


        return (
            x,
            video,
            y,
            location
        )


print("Multimodal dataset ready.")

Multimodal dataset ready.


In [15]:
# ============================================================
# CELL 10: NORMALIZATION
# ============================================================

def get_normalization(samples):

    data = np.concatenate(
        [
            s["x"]
            for s in samples
        ],
        axis=0
    )


    mean = data.mean(
        axis=0
    )


    std = (
        data.std(axis=0)
        + 1e-8
    )


    return mean, std


print("Normalization function ready.")

Normalization function ready.


In [16]:
# ============================================================
# CELL 11: CLASS WEIGHTS
# ============================================================

def get_class_weights(
    train_samples
):

    labels = np.array(
        [
            s["y"]
            for s in train_samples
        ]
    )


    counts = np.bincount(
        labels,
        minlength=NUM_CLASSES
    ).astype(np.float32)


    weights = (
        counts.sum()
        /
        (counts + 1e-6)
    )


    weights = (
        weights
        /
        weights.mean()
    )


    weights = np.clip(
        weights,
        0.1,
        10.0
    )


    return torch.tensor(
        weights,
        dtype=torch.float32,
        device=DEVICE
    )


print("Class-weight function ready.")

Class-weight function ready.


In [18]:
# ============================================================
# CELL 12: EXP11 CROSS-MODAL ATTENTION MODEL
# ============================================================

class CrossModalAttentionWEAR(nn.Module):

    def __init__(
        self,
        num_classes=19
    ):

        super().__init__()


        # ====================================================
        # INERTIAL STEM
        # ====================================================

        self.inertial_stem = nn.Sequential(

            nn.Conv1d(
                3,
                32,
                kernel_size=7,
                padding=3,
                bias=False
            ),

            nn.BatchNorm1d(32),

            nn.ReLU(
                inplace=True
            )
        )


        # ====================================================
        # INERTIAL BLOCK 1
        # ====================================================

        self.inertial_block1 = nn.Sequential(

            nn.Conv1d(
                32,
                32,
                kernel_size=5,
                padding=2,
                bias=False
            ),

            nn.BatchNorm1d(32),

            nn.ReLU(
                inplace=True
            ),


            nn.Conv1d(
                32,
                32,
                kernel_size=5,
                padding=2,
                bias=False
            ),

            nn.BatchNorm1d(32),

            nn.ReLU(
                inplace=True
            )
        )


        # ====================================================
        # INERTIAL BLOCK 2
        # ====================================================

        self.inertial_block2 = nn.Sequential(

            nn.Conv1d(
                32,
                64,
                kernel_size=5,
                stride=2,
                padding=2,
                bias=False
            ),

            nn.BatchNorm1d(64),

            nn.ReLU(
                inplace=True
            ),


            nn.Conv1d(
                64,
                64,
                kernel_size=5,
                padding=2,
                bias=False
            ),

            nn.BatchNorm1d(64),

            nn.ReLU(
                inplace=True
            )
        )


        # ====================================================
        # INERTIAL BLOCK 3
        # ====================================================

        self.inertial_block3 = nn.Sequential(

            nn.Conv1d(
                64,
                128,
                kernel_size=5,
                stride=2,
                padding=2,
                bias=False
            ),

            nn.BatchNorm1d(128),

            nn.ReLU(
                inplace=True
            ),


            nn.Conv1d(
                128,
                128,
                kernel_size=5,
                padding=2,
                bias=False
            ),

            nn.BatchNorm1d(128),

            nn.ReLU(
                inplace=True
            )
        )


        # ====================================================
        # VIDEO PROJECTION
        # ====================================================

        self.video_projection = nn.Sequential(

            nn.Linear(
                768,
                256
            ),

            nn.LayerNorm(256),

            nn.ReLU(
                inplace=True
            ),

            nn.Dropout(0.2)
        )


        # ====================================================
        # CROSS-MODAL ATTENTION
        # ====================================================

        self.inertial_query = nn.Linear(
            128,
            128
        )


        self.video_key = nn.Linear(
            256,
            128
        )


        self.video_value = nn.Linear(
            256,
            128
        )


        self.attention_scale = (
            128 ** -0.5
        )


        # ====================================================
        # SENSOR LOCATION
        # ====================================================

        self.location_embedding = nn.Embedding(
            4,
            16
        )


        # ====================================================
        # FUSION
        # ====================================================

        self.fusion = nn.Sequential(

            nn.Linear(
                272,
                256
            ),

            nn.LayerNorm(256),

            nn.ReLU(
                inplace=True
            ),

            nn.Dropout(0.4),


            nn.Linear(
                256,
                128
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Dropout(0.3)
        )


        # ====================================================
        # CLASSIFIER
        # ====================================================

        self.classifier = nn.Linear(
            128,
            num_classes
        )


    def forward(
        self,
        inertial,
        video,
        location
    ):

        # ====================================================
        # INERTIAL FEATURE
        # ====================================================

        h = self.inertial_stem(
            inertial
        )

        h = self.inertial_block1(
            h
        )

        h = self.inertial_block2(
            h
        )

        h = self.inertial_block3(
            h
        )


        inertial_feature = h.mean(
            dim=2
        )


        # ====================================================
        # VIDEO FEATURE
        # ====================================================

        video_feature = self.video_projection(
            video
        )


        # ====================================================
        # QUERY
        # ====================================================

        query = self.inertial_query(
            inertial_feature
        ).unsqueeze(1)


        # ====================================================
        # KEY + VALUE
        # ====================================================

        key = self.video_key(
            video_feature
        )

        value = self.video_value(
            video_feature
        )


        # ====================================================
        # ATTENTION
        # ====================================================

        attention_scores = torch.matmul(
            query,
            key.transpose(1, 2)
        ) * self.attention_scale


        attention_weights = torch.softmax(
            attention_scores,
            dim=-1
        )


        attended_video = torch.matmul(
            attention_weights,
            value
        ).squeeze(1)


        # ====================================================
        # LOCATION
        # ====================================================

        location_feature = self.location_embedding(
            location
        )


        # ====================================================
        # FUSION
        # ====================================================

        fused = torch.cat(
            [
                inertial_feature,
                attended_video,
                location_feature
            ],
            dim=1
        )


        fused = self.fusion(
            fused
        )


        # ====================================================
        # CLASSIFIER
        # ====================================================

        return self.classifier(
            fused
        )


print(" model created.")

 model created.


In [20]:
# ============================================================
# CELL 13: MODEL SHAPE TEST
# ============================================================

model_test = CrossModalAttentionWEAR(
    NUM_CLASSES
)


dummy_inertial = torch.randn(
    4,
    3,
    50
)


dummy_video = torch.randn(
    4,
    15,
    768
)


dummy_location = torch.randint(
    0,
    4,
    (4,)
)


with torch.no_grad():

    dummy_output = model_test(
        dummy_inertial,
        dummy_video,
        dummy_location
    )


print(
    "Input inertial:",
    dummy_inertial.shape
)

print(
    "Input video:",
    dummy_video.shape
)

print(
    "Output:",
    dummy_output.shape
)


assert dummy_output.shape == (
    4,
    NUM_CLASSES
)


print("\n✅  forward test passed")

Input inertial: torch.Size([4, 3, 50])
Input video: torch.Size([4, 15, 768])
Output: torch.Size([4, 19])

✅  forward test passed


In [21]:
# ============================================================
# CELL 14: TRAINING EPOCH
# ============================================================

def run_epoch(
    model,
    loader,
    loss_function,
    optimizer=None
):

    is_training = (
        optimizer is not None
    )


    if is_training:
        model.train()
    else:
        model.eval()


    total_loss = 0.0

    predictions = []
    targets = []


    with torch.set_grad_enabled(
        is_training
    ):

        for (
            inertial,
            video,
            y,
            location
        ) in loader:


            inertial = inertial.to(
                DEVICE
            )

            video = video.to(
                DEVICE
            )

            y = y.to(
                DEVICE
            )

            location = location.to(
                DEVICE
            )


            # ------------------------------------------------
            # Forward
            # ------------------------------------------------

            logits = model(
                inertial,
                video,
                location
            )


            loss = loss_function(
                logits,
                y
            )


            # ------------------------------------------------
            # Backpropagation
            # ------------------------------------------------

            if is_training:

                optimizer.zero_grad(
                    set_to_none=True
                )

                loss.backward()


                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=5.0
                )


                optimizer.step()


            # ------------------------------------------------
            # Metrics
            # ------------------------------------------------

            total_loss += (
                loss.item()
                * inertial.size(0)
            )


            predictions.append(
                logits.argmax(
                    dim=1
                )
                .detach()
                .cpu()
                .numpy()
            )


            targets.append(
                y.detach()
                .cpu()
                .numpy()
            )


    predictions = np.concatenate(
        predictions
    )

    targets = np.concatenate(
        targets
    )


    avg_loss = (
        total_loss
        /
        len(targets)
    )


    accuracy = accuracy_score(
        targets,
        predictions
    )


    macro_f1 = f1_score(
        targets,
        predictions,
        average="macro"
    )


    return (
        avg_loss,
        accuracy,
        macro_f1
    )


print("Training function ready.")

Training function ready.


In [22]:
# ============================================================
# CELL 15: 5-FOLD SUBJECT-LEVEL SPLIT
# ============================================================

groups = np.array(
    [
        s["sbj_id"]
        for s in samples
    ]
)


y_all = np.array(
    [
        s["y"]
        for s in samples
    ]
)


print(
    "Total samples:",
    len(samples)
)


print(
    "Unique subjects:",
    len(np.unique(groups))
)


gkf = GroupKFold(
    n_splits=N_FOLDS
)


fold_splits = list(
    gkf.split(
        samples,
        y_all,
        groups
    )
)


print(
    "\nNumber of folds:",
    len(fold_splits)
)


for i, (
    train_idx,
    val_idx
) in enumerate(
    fold_splits,
    start=1
):

    print(
        f"Fold {i}: "
        f"train={len(train_idx)} | "
        f"val={len(val_idx)} | "
        f"validation subjects="
        f"{sorted(np.unique(groups[val_idx]).tolist())}"
    )


# OOF storage

oof_predictions = np.zeros(
    len(samples),
    dtype=np.int64
)


oof_targets = y_all.copy()


fold_scores = []
fold_models = []
fold_normalizations = []


print("\n✅ 5-fold split ready")

Total samples: 554528
Unique subjects: 22

Number of folds: 5
Fold 1: train=439720 | val=114808 | validation subjects=[5, 14, 17, 21]
Fold 2: train=450580 | val=103948 | validation subjects=[0, 4, 9, 15]
Fold 3: train=437548 | val=116980 | validation subjects=[3, 7, 11, 12, 19]
Fold 4: train=452400 | val=102128 | validation subjects=[1, 8, 10, 16]
Fold 5: train=437864 | val=116664 | validation subjects=[2, 6, 13, 18, 20]

✅ 5-fold split ready


In [23]:
# ============================================================
# CELL 16: TRAIN ONE FOLD
# ============================================================

def train_one_fold(
    train_samples,
    val_samples,
    fold
):

    print()
    print("=" * 70)
    print(f"TRAINING FOLD {fold}")
    print("=" * 70)


    set_seed(
        42 + fold
    )


    # ========================================================
    # NORMALIZATION
    # ========================================================

    mean, std = get_normalization(
        train_samples
    )


    # ========================================================
    # DATASETS
    # ========================================================

    train_dataset = MultimodalWEARDataset(
        train_samples,
        mean,
        std
    )


    val_dataset = MultimodalWEARDataset(
        val_samples,
        mean,
        std
    )


    # ========================================================
    # DATALOADERS
    # ========================================================

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=False,
        drop_last=True
    )


    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )


    # ========================================================
    # MODEL
    # ========================================================

    model = CrossModalAttentionWEAR(
        num_classes=NUM_CLASSES
    ).to(DEVICE)


    # ========================================================
    # CLASS WEIGHTS
    # ========================================================

    class_weights = get_class_weights(
        train_samples
    )


    loss_function = nn.CrossEntropyLoss(
        weight=class_weights
    )


    # ========================================================
    # OPTIMIZER
    # ========================================================

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )


    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS
    )


    # ========================================================
    # BEST MODEL
    # ========================================================

    best_f1 = -1

    best_state = None

    history = []


    # ========================================================
    # EPOCH LOOP
    # ========================================================

    for epoch in range(EPOCHS):

        train_loss, train_acc, train_f1 = run_epoch(
            model,
            train_loader,
            loss_function,
            optimizer
        )


        val_loss, val_acc, val_f1 = run_epoch(
            model,
            val_loader,
            loss_function
        )


        scheduler.step()


        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_f1": train_f1,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_f1": val_f1
        })


        print(
            f"Epoch {epoch+1:02d}/{EPOCHS} | "
            f"Train F1: {train_f1:.4f} | "
            f"Val F1: {val_f1:.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )


        # ----------------------------------------------------
        # Save best state
        # ----------------------------------------------------

        if val_f1 > best_f1:

            best_f1 = val_f1

            best_state = {
                k: v.detach()
                .cpu()
                .clone()

                for k, v in model.state_dict().items()
            }


    # ========================================================
    # RESTORE BEST MODEL
    # ========================================================

    model.load_state_dict(
        best_state
    )


    # ========================================================
    # SAVE CHECKPOINT
    # ========================================================

    checkpoint_path = os.path.join(
        OUT_DIR,
        f"{CHECKPOINT_PREFIX}_fold{fold}.pt"
    )


    torch.save(
        {
            "model_state_dict":
                model.state_dict(),

            "mean":
                mean,

            "std":
                std,

            "best_f1":
                best_f1,

            "fold":
                fold
        },
        checkpoint_path
    )


    print()
    print(
        f"✅ Fold {fold} best F1: "
        f"{best_f1:.4f}"
    )


    print(
        "Checkpoint:",
        checkpoint_path
    )


    return (
        model,
        (mean, std),
        best_f1,
        history
    )


print("Fold training function ready.")

Fold training function ready.


In [24]:
# ============================================================
# CELL 17: TRAIN FOLD 1
# ============================================================

fold = 1

train_idx, val_idx = fold_splits[fold - 1]


train_samples = [
    samples[i]
    for i in train_idx
]


val_samples = [
    samples[i]
    for i in val_idx
]


model, normalization, best_f1, history = train_one_fold(
    train_samples,
    val_samples,
    fold
)


fold_models.append(model)
fold_normalizations.append(normalization)
fold_scores.append(best_f1)


# ------------------------------------------------------------
# OOF prediction
# ------------------------------------------------------------

mean, std = normalization

val_dataset = MultimodalWEARDataset(
    val_samples,
    mean,
    std
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)


model.eval()

fold_predictions = []


with torch.no_grad():

    for (
        inertial,
        video,
        y,
        location
    ) in val_loader:

        inertial = inertial.to(DEVICE)
        video = video.to(DEVICE)
        location = location.to(DEVICE)

        logits = model(
            inertial,
            video,
            location
        )

        preds = (
            logits.argmax(dim=1)
            .cpu()
            .numpy()
        )

        fold_predictions.append(
            preds
        )


oof_predictions[val_idx] = np.concatenate(
    fold_predictions
)


print("\n✅ Fold 1 complete")


TRAINING FOLD 1
Epoch 01/30 | Train F1: 0.8156 | Val F1: 0.6023 | Val Acc: 0.6304
Epoch 02/30 | Train F1: 0.9092 | Val F1: 0.5699 | Val Acc: 0.6160
Epoch 03/30 | Train F1: 0.9311 | Val F1: 0.5899 | Val Acc: 0.6451
Epoch 04/30 | Train F1: 0.9420 | Val F1: 0.5985 | Val Acc: 0.6432
Epoch 05/30 | Train F1: 0.9507 | Val F1: 0.6085 | Val Acc: 0.6603
Epoch 06/30 | Train F1: 0.9568 | Val F1: 0.6049 | Val Acc: 0.6643
Epoch 07/30 | Train F1: 0.9630 | Val F1: 0.5893 | Val Acc: 0.6688
Epoch 08/30 | Train F1: 0.9660 | Val F1: 0.5760 | Val Acc: 0.6537
Epoch 09/30 | Train F1: 0.9705 | Val F1: 0.5780 | Val Acc: 0.6646
Epoch 10/30 | Train F1: 0.9736 | Val F1: 0.5694 | Val Acc: 0.6480
Epoch 11/30 | Train F1: 0.9763 | Val F1: 0.5842 | Val Acc: 0.6547
Epoch 12/30 | Train F1: 0.9789 | Val F1: 0.5685 | Val Acc: 0.6505
Epoch 13/30 | Train F1: 0.9812 | Val F1: 0.5764 | Val Acc: 0.6558
Epoch 14/30 | Train F1: 0.9838 | Val F1: 0.5698 | Val Acc: 0.6615
Epoch 15/30 | Train F1: 0.9855 | Val F1: 0.5787 | Val Acc: 

In [25]:
# ============================================================
# CELL 18: TRAIN FOLD 2
# ============================================================

fold = 2

train_idx, val_idx = fold_splits[fold - 1]

train_samples = [
    samples[i]
    for i in train_idx
]

val_samples = [
    samples[i]
    for i in val_idx
]


model, normalization, best_f1, history = train_one_fold(
    train_samples,
    val_samples,
    fold
)


fold_models.append(model)
fold_normalizations.append(normalization)
fold_scores.append(best_f1)


mean, std = normalization

val_dataset = MultimodalWEARDataset(
    val_samples,
    mean,
    std
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)


model.eval()

fold_predictions = []


with torch.no_grad():

    for (
        inertial,
        video,
        y,
        location
    ) in val_loader:

        inertial = inertial.to(DEVICE)
        video = video.to(DEVICE)
        location = location.to(DEVICE)

        logits = model(
            inertial,
            video,
            location
        )

        fold_predictions.append(
            logits.argmax(
                dim=1
            )
            .cpu()
            .numpy()
        )


oof_predictions[val_idx] = np.concatenate(
    fold_predictions
)


print("\n✅ Fold 2 complete")


TRAINING FOLD 2
Epoch 01/30 | Train F1: 0.8153 | Val F1: 0.6049 | Val Acc: 0.6440
Epoch 02/30 | Train F1: 0.9117 | Val F1: 0.6001 | Val Acc: 0.6035
Epoch 03/30 | Train F1: 0.9317 | Val F1: 0.5946 | Val Acc: 0.6415
Epoch 04/30 | Train F1: 0.9444 | Val F1: 0.5917 | Val Acc: 0.6356
Epoch 05/30 | Train F1: 0.9515 | Val F1: 0.5866 | Val Acc: 0.6211
Epoch 06/30 | Train F1: 0.9587 | Val F1: 0.5764 | Val Acc: 0.6271
Epoch 07/30 | Train F1: 0.9639 | Val F1: 0.5857 | Val Acc: 0.6208
Epoch 08/30 | Train F1: 0.9679 | Val F1: 0.5798 | Val Acc: 0.6269
Epoch 09/30 | Train F1: 0.9714 | Val F1: 0.5752 | Val Acc: 0.6233
Epoch 10/30 | Train F1: 0.9747 | Val F1: 0.5681 | Val Acc: 0.6262
Epoch 11/30 | Train F1: 0.9770 | Val F1: 0.5729 | Val Acc: 0.6319
Epoch 12/30 | Train F1: 0.9797 | Val F1: 0.5776 | Val Acc: 0.6333
Epoch 13/30 | Train F1: 0.9815 | Val F1: 0.5459 | Val Acc: 0.6118
Epoch 14/30 | Train F1: 0.9844 | Val F1: 0.5688 | Val Acc: 0.6222
Epoch 15/30 | Train F1: 0.9860 | Val F1: 0.5591 | Val Acc: 

In [26]:
# ============================================================
# CELL 19: TRAIN FOLD 3
# ============================================================

fold = 3

train_idx, val_idx = fold_splits[fold - 1]

train_samples = [
    samples[i]
    for i in train_idx
]

val_samples = [
    samples[i]
    for i in val_idx
]


model, normalization, best_f1, history = train_one_fold(
    train_samples,
    val_samples,
    fold
)


fold_models.append(model)
fold_normalizations.append(normalization)
fold_scores.append(best_f1)


mean, std = normalization

val_dataset = MultimodalWEARDataset(
    val_samples,
    mean,
    std
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)


model.eval()

fold_predictions = []


with torch.no_grad():

    for (
        inertial,
        video,
        y,
        location
    ) in val_loader:

        inertial = inertial.to(DEVICE)
        video = video.to(DEVICE)
        location = location.to(DEVICE)

        logits = model(
            inertial,
            video,
            location
        )

        fold_predictions.append(
            logits.argmax(
                dim=1
            )
            .cpu()
            .numpy()
        )


oof_predictions[val_idx] = np.concatenate(
    fold_predictions
)


print("\n✅ Fold 3 complete")


TRAINING FOLD 3
Epoch 01/30 | Train F1: 0.8136 | Val F1: 0.6474 | Val Acc: 0.6673
Epoch 02/30 | Train F1: 0.9092 | Val F1: 0.6386 | Val Acc: 0.6780
Epoch 03/30 | Train F1: 0.9298 | Val F1: 0.6231 | Val Acc: 0.6882
Epoch 04/30 | Train F1: 0.9424 | Val F1: 0.6324 | Val Acc: 0.6808
Epoch 05/30 | Train F1: 0.9506 | Val F1: 0.6164 | Val Acc: 0.6837
Epoch 06/30 | Train F1: 0.9574 | Val F1: 0.6216 | Val Acc: 0.6735
Epoch 07/30 | Train F1: 0.9640 | Val F1: 0.6005 | Val Acc: 0.6721
Epoch 08/30 | Train F1: 0.9676 | Val F1: 0.6021 | Val Acc: 0.6648
Epoch 09/30 | Train F1: 0.9717 | Val F1: 0.5939 | Val Acc: 0.6622
Epoch 10/30 | Train F1: 0.9743 | Val F1: 0.6177 | Val Acc: 0.6833
Epoch 11/30 | Train F1: 0.9773 | Val F1: 0.6050 | Val Acc: 0.6491
Epoch 12/30 | Train F1: 0.9799 | Val F1: 0.5970 | Val Acc: 0.6730
Epoch 13/30 | Train F1: 0.9821 | Val F1: 0.5900 | Val Acc: 0.6681
Epoch 14/30 | Train F1: 0.9845 | Val F1: 0.5780 | Val Acc: 0.6631
Epoch 15/30 | Train F1: 0.9864 | Val F1: 0.5903 | Val Acc: 

In [27]:
# ============================================================
# CELL 20: TRAIN FOLD 4
# ============================================================

fold = 4

train_idx, val_idx = fold_splits[fold - 1]

train_samples = [
    samples[i]
    for i in train_idx
]

val_samples = [
    samples[i]
    for i in val_idx
]


model, normalization, best_f1, history = train_one_fold(
    train_samples,
    val_samples,
    fold
)


fold_models.append(model)
fold_normalizations.append(normalization)
fold_scores.append(best_f1)


mean, std = normalization

val_dataset = MultimodalWEARDataset(
    val_samples,
    mean,
    std
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)


model.eval()

fold_predictions = []


with torch.no_grad():

    for (
        inertial,
        video,
        y,
        location
    ) in val_loader:

        inertial = inertial.to(DEVICE)
        video = video.to(DEVICE)
        location = location.to(DEVICE)

        logits = model(
            inertial,
            video,
            location
        )

        fold_predictions.append(
            logits.argmax(
                dim=1
            )
            .cpu()
            .numpy()
        )


oof_predictions[val_idx] = np.concatenate(
    fold_predictions
)


print("\n✅ Fold 4 complete")


TRAINING FOLD 4
Epoch 01/30 | Train F1: 0.8200 | Val F1: 0.5967 | Val Acc: 0.6093
Epoch 02/30 | Train F1: 0.9115 | Val F1: 0.5939 | Val Acc: 0.6390
Epoch 03/30 | Train F1: 0.9306 | Val F1: 0.6087 | Val Acc: 0.6343
Epoch 04/30 | Train F1: 0.9428 | Val F1: 0.5723 | Val Acc: 0.6121
Epoch 05/30 | Train F1: 0.9506 | Val F1: 0.5824 | Val Acc: 0.6351
Epoch 06/30 | Train F1: 0.9577 | Val F1: 0.5980 | Val Acc: 0.6565
Epoch 07/30 | Train F1: 0.9627 | Val F1: 0.6016 | Val Acc: 0.6568
Epoch 08/30 | Train F1: 0.9671 | Val F1: 0.5824 | Val Acc: 0.6391
Epoch 09/30 | Train F1: 0.9710 | Val F1: 0.5774 | Val Acc: 0.6133
Epoch 10/30 | Train F1: 0.9737 | Val F1: 0.6109 | Val Acc: 0.6730
Epoch 11/30 | Train F1: 0.9763 | Val F1: 0.5954 | Val Acc: 0.6629
Epoch 12/30 | Train F1: 0.9788 | Val F1: 0.5843 | Val Acc: 0.6566
Epoch 13/30 | Train F1: 0.9815 | Val F1: 0.5874 | Val Acc: 0.6562
Epoch 14/30 | Train F1: 0.9835 | Val F1: 0.5851 | Val Acc: 0.6505
Epoch 15/30 | Train F1: 0.9856 | Val F1: 0.5898 | Val Acc: 

In [28]:
# ============================================================
# CELL 21: TRAIN FOLD 5
# ============================================================

fold = 5

train_idx, val_idx = fold_splits[fold - 1]

train_samples = [
    samples[i]
    for i in train_idx
]

val_samples = [
    samples[i]
    for i in val_idx
]


model, normalization, best_f1, history = train_one_fold(
    train_samples,
    val_samples,
    fold
)


fold_models.append(model)
fold_normalizations.append(normalization)
fold_scores.append(best_f1)


mean, std = normalization

val_dataset = MultimodalWEARDataset(
    val_samples,
    mean,
    std
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)


model.eval()

fold_predictions = []


with torch.no_grad():

    for (
        inertial,
        video,
        y,
        location
    ) in val_loader:

        inertial = inertial.to(DEVICE)
        video = video.to(DEVICE)
        location = location.to(DEVICE)

        logits = model(
            inertial,
            video,
            location
        )

        fold_predictions.append(
            logits.argmax(
                dim=1
            )
            .cpu()
            .numpy()
        )


oof_predictions[val_idx] = np.concatenate(
    fold_predictions
)


print("\n✅ Fold 5 complete")


TRAINING FOLD 5
Epoch 01/30 | Train F1: 0.8108 | Val F1: 0.6473 | Val Acc: 0.6835
Epoch 02/30 | Train F1: 0.9060 | Val F1: 0.6522 | Val Acc: 0.6671
Epoch 03/30 | Train F1: 0.9265 | Val F1: 0.6519 | Val Acc: 0.7092
Epoch 04/30 | Train F1: 0.9399 | Val F1: 0.6698 | Val Acc: 0.7126
Epoch 05/30 | Train F1: 0.9480 | Val F1: 0.6424 | Val Acc: 0.6901
Epoch 06/30 | Train F1: 0.9557 | Val F1: 0.6594 | Val Acc: 0.7125
Epoch 07/30 | Train F1: 0.9608 | Val F1: 0.6548 | Val Acc: 0.7112
Epoch 08/30 | Train F1: 0.9662 | Val F1: 0.6350 | Val Acc: 0.7015
Epoch 09/30 | Train F1: 0.9684 | Val F1: 0.6663 | Val Acc: 0.7147
Epoch 10/30 | Train F1: 0.9724 | Val F1: 0.6524 | Val Acc: 0.7047
Epoch 11/30 | Train F1: 0.9763 | Val F1: 0.6485 | Val Acc: 0.7127
Epoch 12/30 | Train F1: 0.9784 | Val F1: 0.6752 | Val Acc: 0.7300
Epoch 13/30 | Train F1: 0.9811 | Val F1: 0.6384 | Val Acc: 0.7030
Epoch 14/30 | Train F1: 0.9832 | Val F1: 0.6640 | Val Acc: 0.7149
Epoch 15/30 | Train F1: 0.9859 | Val F1: 0.6405 | Val Acc: 

In [29]:
# ============================================================
# CELL 22: EXP11 RESULTS
# ============================================================

mean_cv_f1 = np.mean(
    fold_scores
)

std_cv_f1 = np.std(
    fold_scores
)


oof_f1 = f1_score(
    oof_targets,
    oof_predictions,
    average="macro"
)


oof_accuracy = accuracy_score(
    oof_targets,
    oof_predictions
)


print("=" * 70)
print("EXPERIMENT 11 RESULTS")
print("=" * 70)


print(
    "Fold scores:",
    [
        round(x, 4)
        for x in fold_scores
    ]
)


print(
    f"\nMean CV Macro-F1 : "
    f"{mean_cv_f1:.4f}"
)


print(
    f"Std CV Macro-F1  : "
    f"{std_cv_f1:.4f}"
)


print(
    f"OOF Macro-F1     : "
    f"{oof_f1:.4f}"
)


print(
    f"OOF Accuracy     : "
    f"{oof_accuracy:.4f}"
)


print("=" * 70)

EXPERIMENT 11 RESULTS
Fold scores: [0.6085, 0.6049, 0.6474, 0.6109, 0.6752]

Mean CV Macro-F1 : 0.6294
Std CV Macro-F1  : 0.0276
OOF Macro-F1     : 0.6316
OOF Accuracy     : 0.6757


In [30]:
# ============================================================
# CELL 23: PER-CLASS F1
# ============================================================

id_to_label = {
    value: key
    for key, value in LABEL_MAP.items()
}


target_names = [
    id_to_label[i]
    for i in range(NUM_CLASSES)
]


print(
    classification_report(
        oof_targets,
        oof_predictions,
        target_names=target_names,
        digits=4
    )
)

                              precision    recall  f1-score   support

                        null     0.7323    0.7490    0.7406    220456
                     jogging     0.7723    0.8968    0.8299     20024
     jogging (rotating arms)     0.7966    0.8018    0.7992     17536
          jogging (skipping)     0.7133    0.7924    0.7508     18212
         jogging (sidesteps)     0.7708    0.7786    0.7747     19732
        jogging (butt-kicks)     0.7665    0.6958    0.7295     17224
        stretching (triceps)     0.5327    0.4801    0.5051     18636
        stretching (lunging)     0.3425    0.3024    0.3212     19524
      stretching (shoulders)     0.4759    0.3500    0.4034     18052
     stretching (hamstrings)     0.7035    0.7662    0.7335     18412
stretching (lumbar rotation)     0.7297    0.7585    0.7438     18584
                    push-ups     0.5130    0.5146    0.5138     17308
          push-ups (complex)     0.6117    0.5099    0.5562     18520
                   

In [31]:
# ============================================================
# CELL 24: LOAD TEST DATA
# ============================================================

print("=" * 70)
print("LOADING TEST DATA")
print("=" * 70)


test_x = np.load(
    TEST_INERTIAL_PATH
).astype(
    np.float32
)


test_video = np.load(
    TEST_VIDEO_PATH
)


test_meta = pd.read_csv(
    TEST_META_PATH
)


print(
    "Test inertial shape:",
    test_x.shape
)


print(
    "Test video ORIGINAL shape:",
    test_video.shape
)


print(
    "Test metadata shape:",
    test_meta.shape
)


# ============================================================
# IMPORTANT TEST VIDEOMAE FIX
# ============================================================

if test_video.ndim == 3:

    # Expected:
    # (N, 768, 15)
    #
    # Model expects:
    # (N, 15, 768)

    if (
        test_video.shape[1] == 768
        and test_video.shape[2] == 15
    ):

        test_video = np.transpose(
            test_video,
            (0, 2, 1)
        )


print(
    "Test video FINAL shape:",
    test_video.shape
)


assert test_x.shape[0] == test_video.shape[0]

assert test_video.shape[1:] == (
    15,
    768
)


print("\n✅ Test shapes are correct")

LOADING TEST DATA
Test inertial shape: (12234, 50, 3)
Test video ORIGINAL shape: (12234, 768, 15)
Test metadata shape: (12234, 3)
Test video FINAL shape: (12234, 15, 768)

✅ Test shapes are correct


In [32]:
# ============================================================
# CELL 25: FINAL TEST DATASET
# ============================================================

class FinalTestDataset(Dataset):

    def __init__(
        self,
        x,
        video,
        meta,
        mean,
        std
    ):

        self.x = x
        self.video = video

        self.meta = meta.reset_index(
            drop=True
        )

        self.mean = mean
        self.std = std


    def __len__(self):

        return len(self.x)


    def __getitem__(self, index):

        # ----------------------------------------------------
        # Inertial
        # ----------------------------------------------------

        x = self.x[index].copy()


        x = (
            x - self.mean
        ) / self.std


        # [50,3] -> [3,50]

        x = torch.from_numpy(
            x.T.copy()
        ).float()


        # ----------------------------------------------------
        # Video
        # ----------------------------------------------------

        video = torch.from_numpy(
            np.asarray(
                self.video[index]
            )
        ).float()


        # ----------------------------------------------------
        # Location
        # ----------------------------------------------------

        if "sensor_location" in self.meta.columns:

            location_name = self.meta.loc[
                index,
                "sensor_location"
            ]

        else:

            location_name = self.meta.loc[
                index,
                "inertial_sensor_location"
            ]


        location = torch.tensor(
            LOCATION_MAP[
                location_name
            ],
            dtype=torch.long
        )


        # ----------------------------------------------------
        # ID
        # ----------------------------------------------------

        sample_id = int(
            self.meta.loc[
                index,
                "id"
            ]
        )


        return (
            x,
            video,
            location,
            sample_id
        )


print("Final test dataset ready.")

Final test dataset ready.


In [33]:
# ============================================================
# CELL 26: CHECK CHECKPOINTS
# ============================================================

checkpoint_paths = [
    os.path.join(
        OUT_DIR,
        f"{CHECKPOINT_PREFIX}_fold{fold}.pt"
    )
    for fold in range(1, 6)
]


print("Required checkpoints:")


for path in checkpoint_paths:

    exists = os.path.exists(
        path
    )

    print(
        os.path.basename(path),
        "->",
        "FOUND" if exists else "MISSING"
    )


missing = [
    path
    for path in checkpoint_paths
    if not os.path.exists(path)
]


if missing:

    raise FileNotFoundError(
        "One or more Exp11 checkpoints are missing."
    )


print(
    "\n✅ All 5 checkpoints found"
)

Required checkpoints:
exp11_crossmodal_fold1.pt -> FOUND
exp11_crossmodal_fold2.pt -> FOUND
exp11_crossmodal_fold3.pt -> FOUND
exp11_crossmodal_fold4.pt -> FOUND
exp11_crossmodal_fold5.pt -> FOUND

✅ All 5 checkpoints found


In [34]:
# ============================================================
# CELL 27: 5-FOLD ENSEMBLE INFERENCE
# ============================================================

all_fold_probs = []

all_fold_ids = []


for fold, checkpoint_path in enumerate(
    checkpoint_paths,
    start=1
):

    print()
    print("=" * 60)
    print(
        f"LOADING FOLD {fold}"
    )
    print("=" * 60)


    # --------------------------------------------------------
    # IMPORTANT PYTORCH 2.6 FIX
    # --------------------------------------------------------

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False
    )


    mean = checkpoint["mean"]
    std = checkpoint["std"]


    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = CrossModalAttentionWEAR(
        NUM_CLASSES
    ).to(DEVICE)


    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )


    model.eval()


    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------

    test_dataset = FinalTestDataset(
        test_x,
        test_video,
        test_meta,
        mean,
        std
    )


    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )


    fold_probs = []
    fold_ids = []


    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------

    with torch.no_grad():

        for (
            inertial,
            video,
            location,
            sample_id
        ) in test_loader:


            inertial = inertial.to(
                DEVICE
            )

            video = video.to(
                DEVICE
            )

            location = location.to(
                DEVICE
            )


            logits = model(
                inertial,
                video,
                location
            )


            probabilities = torch.softmax(
                logits,
                dim=1
            )


            # IMPORTANT:
            # GPU -> CPU -> NumPy

            fold_probs.append(
                probabilities
                .detach()
                .cpu()
                .numpy()
            )


            fold_ids.append(
                sample_id.numpy()
            )


    fold_probs = np.concatenate(
        fold_probs,
        axis=0
    )


    fold_ids = np.concatenate(
        fold_ids,
        axis=0
    )


    print(
        "Fold probability shape:",
        fold_probs.shape
    )


    all_fold_probs.append(
        fold_probs
    )


    all_fold_ids.append(
        fold_ids
    )


    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print("\n✅ All 5 folds predicted")


LOADING FOLD 1
Fold probability shape: (12234, 19)

LOADING FOLD 2
Fold probability shape: (12234, 19)

LOADING FOLD 3
Fold probability shape: (12234, 19)

LOADING FOLD 4
Fold probability shape: (12234, 19)

LOADING FOLD 5
Fold probability shape: (12234, 19)

✅ All 5 folds predicted


In [35]:
# ============================================================
# CELL 28: ENSEMBLE PROBABILITIES
# ============================================================

ensemble_probs = np.mean(
    np.stack(
        all_fold_probs,
        axis=0
    ),
    axis=0
)


# Use IDs from first fold

test_ids = all_fold_ids[0]


print(
    "Ensemble probability shape:",
    ensemble_probs.shape
)


print(
    "Expected:",
    (
        len(test_x),
        NUM_CLASSES
    )
)


assert ensemble_probs.shape == (
    len(test_x),
    NUM_CLASSES
)


# ------------------------------------------------------------
# Final class prediction
# ------------------------------------------------------------

final_predictions = (
    ensemble_probs.argmax(
        axis=1
    )
)


print(
    "\nFinal prediction shape:",
    final_predictions.shape
)


print(
    "Unique predicted classes:",
    np.unique(
        final_predictions
    )
)


print("\n✅ Ensemble prediction ready")

Ensemble probability shape: (12234, 19)
Expected: (12234, 19)

Final prediction shape: (12234,)
Unique predicted classes: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]

✅ Ensemble prediction ready


In [36]:
# ============================================================
# CELL 29: PREDICTION SANITY CHECK
# ============================================================

print("=" * 70)
print("PREDICTION SANITY CHECK")
print("=" * 70)


print(
    "Number of test samples:",
    len(test_x)
)


print(
    "Number of predictions:",
    len(final_predictions)
)


print(
    "Number of IDs:",
    len(test_ids)
)


assert len(test_x) == len(final_predictions)
assert len(test_ids) == len(final_predictions)


print(
    "\nPrediction class distribution:"
)


unique_classes, counts = np.unique(
    final_predictions,
    return_counts=True
)


for cls, count in zip(
    unique_classes,
    counts
):

    print(
        f"{cls:2d} | "
        f"{LABEL_MAP.keys().__iter__().__next__() if False else id_to_label[cls]} | "
        f"{count}"
    )


print(
    "\nAverage maximum probability:",
    ensemble_probs.max(axis=1).mean()
)


print(
    "Minimum maximum probability:",
    ensemble_probs.max(axis=1).min()
)


print(
    "Maximum maximum probability:",
    ensemble_probs.max(axis=1).max()
)


print("\n✅ Sanity checks passed")

PREDICTION SANITY CHECK
Number of test samples: 12234
Number of predictions: 12234
Number of IDs: 12234

Prediction class distribution:
 0 | null | 6744
 1 | jogging | 424
 2 | jogging (rotating arms) | 327
 3 | jogging (skipping) | 347
 4 | jogging (sidesteps) | 280
 5 | jogging (butt-kicks) | 347
 6 | stretching (triceps) | 95
 7 | stretching (lunging) | 146
 8 | stretching (shoulders) | 374
 9 | stretching (hamstrings) | 251
10 | stretching (lumbar rotation) | 430
11 | push-ups | 28
12 | push-ups (complex) | 394
13 | sit-ups | 272
14 | sit-ups (complex) | 443
15 | burpees | 487
16 | lunges | 263
17 | lunges (complex) | 323
18 | bench-dips | 259

Average maximum probability: 0.74429363
Minimum maximum probability: 0.20127991
Maximum maximum probability: 0.99996156

✅ Sanity checks passed


In [37]:
# ============================================================
# CELL 30: SAMPLE SUBMISSION
# ============================================================

sample_submission = pd.read_csv(
    SAMPLE_SUB_PATH
)


print(
    "Sample submission shape:",
    sample_submission.shape
)


print(
    "\nColumns:"
)


print(
    sample_submission.columns.tolist()
)


print(
    "\nFirst 5 rows:"
)


display(
    sample_submission.head()
)

Sample submission shape: (12234, 2)

Columns:
['id', 'target_feature']

First 5 rows:


,id,target_feature
0,0,7.0
1,1,5.0
2,2,10.0
3,3,4.0
4,4,7.0


In [38]:
# ============================================================
# CELL 31: CREATE FINAL SUBMISSION
# ============================================================

# ------------------------------------------------------------
# Detect target column
# ------------------------------------------------------------

if "id" in sample_submission.columns:

    target_columns = [
        col
        for col in sample_submission.columns
        if col != "id"
    ]

else:

    target_columns = list(
        sample_submission.columns[1:]
    )


if len(target_columns) != 1:

    raise ValueError(
        "Could not uniquely determine "
        "submission target column."
    )


target_col = target_columns[0]


print(
    "Submission target column:",
    target_col
)


# ------------------------------------------------------------
# Build submission
# ------------------------------------------------------------

submission = pd.DataFrame({

    "id": test_ids,

    target_col:
        final_predictions
})


# ------------------------------------------------------------
# Sort by ID
# ------------------------------------------------------------

submission = (
    submission
    .sort_values("id")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

assert len(submission) == len(test_meta)

assert submission["id"].is_unique

assert submission[target_col].isna().sum() == 0


print(
    "\nSubmission shape:",
    submission.shape
)


print(
    "\nSubmission preview:"
)


display(
    submission.head(10)
)

Submission target column: target_feature

Submission shape: (12234, 2)

Submission preview:


,id,target_feature
0,0,14
1,1,0
2,2,1
3,3,0
4,4,7
5,5,0
6,6,0
7,7,0
8,8,0
9,9,0


In [39]:
# ============================================================
# CELL 32: SAVE FINAL SUBMISSION
# ============================================================

submission_path = (
    "/kaggle/working/"
    "exp11_submission.csv"
)


submission.to_csv(
    submission_path,
    index=False
)


print("=" * 70)
print("FINAL SUBMISSION CREATED")
print("=" * 70)


print(
    "Path:",
    submission_path
)


print(
    "Shape:",
    submission.shape
)


print(
    "\nFile exists:",
    os.path.exists(
        submission_path
    )
)


print(
    "\nFirst 10 rows:"
)


display(
    submission.head(10)
)


print(
    "\nLast 10 rows:"
)


display(
    submission.tail(10)
)


print(
    "\n🎉  PIPELINE COMPLETE"
)

FINAL SUBMISSION CREATED
Path: /kaggle/working/exp11_submission.csv
Shape: (12234, 2)

File exists: True

First 10 rows:


,id,target_feature
0,0,14
1,1,0
2,2,1
3,3,0
4,4,7
5,5,0
6,6,0
7,7,0
8,8,0
9,9,0



Last 10 rows:


,id,target_feature
12224,12224,0
12225,12225,0
12226,12226,2
12227,12227,0
12228,12228,0
12229,12229,0
12230,12230,14
12231,12231,0
12232,12232,10
12233,12233,0



🎉 EXP11 PIPELINE COMPLETE


                    ┌─────────────────────────────┐
                    │       INPUT DATA            │
                    └──────────────┬──────────────┘
                                   │
                  ┌────────────────┴────────────────┐
                  │                                 │
                  ▼                                 ▼
        INERTIAL SENSOR DATA                  VIDEOMAE FEATURES
          Shape: [3, 50]                       Shape: [15, 768]
                  │                                 │
                  │                                 ▼
                  │                       Linear 768 → 256
                  │                       LayerNorm
                  │                       ReLU
                  │                       Dropout 0.2
                  │                                 │
                  │                                 ▼
                  │                         Video features
                  │                           [15, 256]
                  │                                 │
                  │                    ┌────────────┴────────────┐
                  │                    │                         │
                  │                    ▼                         ▼
                  │                 KEY 256→128              VALUE 256→128
                  │                    │                         │
                  │                    └────────────┬────────────┘
                  │                                 │
                  ▼                                 │
          Conv1D 3 → 32                             │
          Kernel = 7                                │
          BatchNorm + ReLU                          │
                  │                                 │
                  ▼                                 │
          Block 1                                    │
          32 → 32                                    │
          Conv5 + BN + ReLU                          │
          Conv5 + BN + ReLU                          │
                  │                                 │
                  ▼                                 │
          Block 2                                    │
          32 → 64                                    │
          Stride = 2                                 │
          Conv5 + BN + ReLU                          │
          Conv5 + BN + ReLU                          │
                  │                                 │
                  ▼                                 │
          Block 3                                    │
          64 → 128                                   │
          Stride = 2                                 │
          Conv5 + BN + ReLU                          │
          Conv5 + BN + ReLU                          │
                  │                                 │
                  ▼                                 │
        Global Mean Pooling                          │
                  │                                 │
                  ▼                                 │
          Inertial feature                           │
             [128]                                   │
                  │                                 │
                  ▼                                 │
          Query 128 → 128                            │
                  │                                 │
                  └───────────────┐                 │
                                  ▼                 ▼
                         CROSS-MODAL ATTENTION
                         
                       Query: Inertial
                       Key:   Video
                       Value: Video

                         Q × Kᵀ / √128
                                │
                                ▼
                           Softmax
                                │
                                ▼
                    Weighted Video Values
                                │
                                ▼
                       Attended Video
                           [128]
                                │
                                │
              ┌─────────────────┴─────────────────┐
              │                                   │
              ▼                                   ▼
       Inertial [128]                       Video [128]
              │                                   │
              └─────────────────┬─────────────────┘
                                │
                                │
                    SENSOR LOCATION EMBEDDING
                            4 → 16
                                │
                                ▼
                         Concatenation
                      128 + 128 + 16 = 272
                                │
                                ▼
                         Linear 272 → 256
                         LayerNorm
                         ReLU
                         Dropout 0.4
                                │
                                ▼
                         Linear 256 → 128
                         ReLU
                         Dropout 0.3
                                │
                                ▼
                         Linear 128 → 19
                                │
                                ▼
                       19 ACTIVITY CLASSES